# 強化学習の考え方と価値関数

強化学習を最初に難しく見せるのは、状態、行動、報酬、価値、方策が一気に出てくるからです。けれど本質はかなり素朴で、「いま少し得か」ではなく「この行動のあと、しばらく先まで見て得か」で意思決定したい、という一点にあります。


## 参考動画（外部）

授業本編ではなく、別の説明で見直したいときの参考材料です。

- [強化学習の基本発想](https://www.youtube.com/watch?v=jwHVLrtkt5w)
- [マルコフ決定過程の導入](https://www.youtube.com/watch?v=R8CyNE8Vgg4)


## まずは「1回の報酬」ではなく「この先の合計」を見る

強化学習では、ある行動がよいかを、その瞬間の報酬だけでは決めません。少し遠回りでも、あとで大きく得するなら、その行動には価値があります。そこで使うのが割引収益 `return` です。`gamma` は「未来をどこまで大事に見るか」を決める係数で、1 に近いほど先の報酬を気にするようになります。


In [ ]:
rewards = [0, 1, 0, 2]
gamma = 0.90
g = 0.0
for r in reversed(rewards):
    g = r + gamma * g
print("task = rl-foundation", "return=", round(g, 6))


この 1 本の数値が、「いまの状態から将来どれだけ得しそうか」の原型です。価値関数は、この考え方を状態ごと、あるいは状態と行動の組ごとに持ち直したものだと考えると自然です。


## 価値関数は、「この先どう動くつもりか」まで含めた長期期待値

同じ状態にいても、そのあと強気に進むのか、安全に進むのかで将来の得は変わります。方策 `\pi` は「その状態で次にどう行動するつもりか」を表す約束事で、状態価値 `V^\pi(s)` は「その約束事に従ったとき、その状態からどれだけ得しそうか」を表します。ここで大事なのは、価値関数が単なる即時報酬の表ではなく、遷移先の価値まで織り込んだ期待値だということです。


In [ ]:
import numpy as np

gamma = 0.9
states = ["s0", "s1"]
P_pi = np.array([
    [0.6, 0.4],
    [0.3, 0.7],
])
R_pi = np.array([0.2, 0.8])


方策を固定したとき、価値関数は

\[V^{\pi}=R_{\pi}+\gamma P_{\pi}V^{\pi}\]

を満たします。言い換えると、「いま受け取れそうな報酬」と「次に行く先でまたどれだけ得しそうか」を足したものが、結局いまの価値そのものになる、という自己参照の式です。


In [ ]:
I = np.eye(len(states))
V_exact = np.linalg.solve(I - gamma * P_pi, R_pi)
print("V^pi (exact) =", {s: round(v, 6) for s, v in zip(states, V_exact)})

V_iter = np.zeros(len(states))
for _ in range(40):
    V_iter = R_pi + gamma * (P_pi @ V_iter)
print("V^pi (iterative) =", {s: round(v, 6) for s, v in zip(states, V_iter)})
print("max abs diff =", round(float(np.max(np.abs(V_iter - V_exact))), 10))


解析解で一気に求めても、ベルマン期待更新を何度も回しても、同じ値へ近づきます。この「未来の見込みを次状態の価値に押し込んで、いまへ戻す」考え方が後続の TD 法や Q 学習の土台です。


## 状態価値 `V` と行動価値 `Q` は、見積もる対象が違う

状態価値は「この状態にいること」の価値、行動価値は「この状態でこの行動を取ること」の価値です。制御問題では最終的に行動を選びたいので `Q` が便利ですが、まず `V` を理解しておくと、なぜ `max` が出てくるのかを落ち着いて読めます。


In [ ]:
gamma = 0.9
Q = {
    ("s0", "left"): 0.3,
    ("s0", "right"): 0.1,
    ("s1", "left"): 0.5,
    ("s1", "right"): 0.7,
}
pi = {"s0": {"left": 0.5, "right": 0.5}}
expectation_backup = sum(pi["s0"][a] * (1.0 + gamma * Q[("s1", a)]) for a in ["left", "right"])
optimal_backup = max(1.0 + gamma * Q[("s1", a)] for a in ["left", "right"])
print("expected backup =", round(expectation_backup, 6))
print("optimal backup  =", round(optimal_backup, 6))


同じ次状態を見ていても、「いつもの行動方針どおり平均を取るか」「その場でいちばん良い行動だけを選ぶか」で値は変わります。これがベルマン期待方程式とベルマン最適方程式の差で、前者は「いまの方針を評価する」ため、後者は「最適に制御する」ための式です。


## ここまでで掴みたいこと

1. 強化学習は、目先の得だけでなく長期 return で意思決定する。
2. 価値関数は、「この先どう動くつもりか」を含んだ期待値である。
3. ベルマン更新は、遠い未来を次状態の価値へ押し込んで考える再帰的な見方である。
4. `V` と `Q` の違いを押さえると、後続の TD 法と制御アルゴリズムが読みやすくなる。
